# Bank Marketing — Data Preprocessing

## 1. Introduction

This notebook implements the baseline preprocessing configuration used for
model development.

- Exclude `duration` and retain the other 19 raw predictors.
- Represent the `pdays = 999` sentinel explicitly.
- Preserve the chronological Train/Validation split.
- Fit preprocessing on Train only.
- Keep Test frozen for final evaluation.

## 2. Load Data & Splits

Load the raw data in source order and reproduce the established chronological
boundaries. Only Train and Validation are prepared in this notebook.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "bank-additional"
    / "bank-additional-full.csv"
)

df = pd.read_csv(RAW_DATA_PATH, sep=";")

train_end = int(len(df) * 0.70)
valid_end = int(len(df) * 0.90)

train_df = df.iloc[:train_end].copy()
valid_df = df.iloc[train_end:valid_end].copy()

In [2]:
target = "y"
excluded_features = ["duration"]
modeling_features = [
    column for column in df.columns
    if column not in [target, *excluded_features]
]

X_train = train_df[modeling_features].copy()
X_valid = valid_df[modeling_features].copy()

target_mapping = {"no": 0, "yes": 1}
y_train = train_df[target].map(target_mapping)
y_valid = valid_df[target].map(target_mapping)

pd.DataFrame(
    {
        "rows": [len(X_train), len(X_valid)],
        "features": [X_train.shape[1], X_valid.shape[1]],
        "positive_rate": [y_train.mean(), y_valid.mean()],
    },
    index=["Train", "Validation"],
)

,rows,features,positive_rate
Train,28831,19,0.055704
Validation,8238,19,0.138626


## 3. Feature Definition

The feature groups are defined explicitly from their preprocessing needs.
`pdays` remains one of the 19 raw inputs, but its sentinel and elapsed-day
information are separated before scaling.

In [3]:
numeric_features = [
    "age",
    "campaign",
    "previous",
    "emp.var.rate",
    "cons.price.idx",
    "cons.conf.idx",
    "euribor3m",
    "nr.employed",
]

categorical_features = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "month",
    "day_of_week",
    "poutcome",
]

special_features = ["pdays"]

pd.Series(
    {
        "numeric": len(numeric_features),
        "categorical": len(categorical_features),
        "sentinel_aware": len(special_features),
        "total_raw": len(numeric_features + categorical_features + special_features),
    },
    name="feature_count",
)

numeric            8
categorical       10
sentinel_aware     1
total_raw         19
Name: feature_count, dtype: int64

## 4. Preprocessing Strategy

| Feature group | Baseline transformation |
|---|---|
| Eight ordinary numerical features | `StandardScaler` |
| `pdays` | Previous-contact indicator + elapsed days, then `StandardScaler` |
| Ten categorical features | `OneHotEncoder(handle_unknown="ignore")` |

For `pdays`, `999` becomes `pdays_previously_contacted = 0` and
`pdays_elapsed = 0`. Values from contacted records become an indicator of `1`
and retain their observed elapsed days. Raw `999` is never treated as an
ordinary elapsed-day value.

The baseline keeps `"unknown"` as a category and does not add imputation,
clipping, rare-category grouping, or other feature engineering.

## 5. Preprocessing Pipeline

### 5.1 Build Preprocessor

`ColumnTransformer` applies each transformation only to its assigned columns.

In [4]:
def split_pdays(values):
    pdays = np.asarray(values).reshape(-1)
    previously_contacted = (pdays != 999).astype(float)
    elapsed_days = np.where(pdays == 999, 0.0, pdays.astype(float))
    return np.column_stack([previously_contacted, elapsed_days])


def pdays_feature_names(transformer, input_features):
    return np.asarray(
        ["pdays_previously_contacted", "pdays_elapsed"],
        dtype=object,
    )


pdays_transformer = Pipeline(
    steps=[
        (
            "derive",
            FunctionTransformer(
                split_pdays,
                validate=False,
                feature_names_out=pdays_feature_names,
            ),
        ),
        ("scale", StandardScaler()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features,
        ),
        ("pdays", pdays_transformer, special_features),
    ],
    remainder="drop",
)

### 5.2 Fit on Train & Transform Validation

The scaler statistics and encoder categories are learned from Train. The same
fitted preprocessor is then applied to Validation without refitting.

In [5]:
X_train_prepared = preprocessor.fit_transform(X_train)
X_valid_prepared = preprocessor.transform(X_valid)

In [6]:
feature_names = preprocessor.get_feature_names_out()

transformation_summary = pd.DataFrame(
    {
        "input_shape": [X_train.shape, X_valid.shape],
        "output_shape": [X_train_prepared.shape, X_valid_prepared.shape],
        "output_type": [
            type(X_train_prepared).__name__,
            type(X_valid_prepared).__name__,
        ],
    },
    index=["Train", "Validation"],
)

display(transformation_summary)
pd.Series(feature_names, name="transformed_feature")

,input_shape,output_shape,output_type
Train,"(28831, 19)","(28831, 62)",ndarray
Validation,"(8238, 19)","(8238, 62)",ndarray


0                              num__age
1                         num__campaign
2                         num__previous
3                     num__emp.var.rate
4                   num__cons.price.idx
                    ...                
57                cat__poutcome_failure
58            cat__poutcome_nonexistent
59                cat__poutcome_success
60    pdays__pdays_previously_contacted
61                 pdays__pdays_elapsed
Name: transformed_feature, Length: 62, dtype: str

The baseline preprocessing produces 62 features: eight scaled numerical
columns, 52 one-hot encoded columns learned from Train, and two sentinel-aware
`pdays` columns.

## 6. Preprocessing Summary

- The 19 prediction-time raw predictors are retained and `duration` is excluded.
- Eight ordinary numerical features use `StandardScaler`.
- Raw `pdays` is replaced by a previous-contact indicator and elapsed days;
  both derived columns are scaled after the sentinel is resolved.
- Categorical features use `OneHotEncoder(handle_unknown="ignore")`.
- The preprocessor is fitted on Train and reused unchanged for Validation.
- No imputation, clipping, category merging, or additional feature engineering
  is applied in the baseline representation.

This representation follows the EDA interpretation of `pdays = 999`. Its model
performance must be established again in the baseline and experiment notebooks;
the earlier raw-`pdays` metrics do not validate this revised configuration.